# 第 2 章 · 三层边界

**这一章你会得到什么**：能准确说出循环里“哪些逻辑属于 Agent、哪些属于 Model、哪些属于 Environment”，并理解为什么用 Protocol（鸭子类型）而不是继承。

> 边界答不清，后面的 Tool、Memory、Sandbox、Evaluation 都会混在一起。

## 📖 对照源码（在 IDE 里打开这些文件，边看边跑）

- `src/minisweagent/__init__.py` **L43–80** — 三个 Protocol（Model / Environment / Agent）
- `src/minisweagent/agents/default.py` **L152–155** — `execute_actions()`（只依赖 env 的能力，不认类）

> 快捷：代码格里 `函数名??` 直接打印源码；或用 `show_source("相对路径", 起始行, 结束行)`。

In [1]:
# 环境自检：把源码目录加入 sys.path，并切到仓库根目录
import os, sys
from pathlib import Path
os.environ["MSWEA_SILENT_STARTUP"] = "1"
REPO = Path(r"/Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent")
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
import minisweagent
print("Python:", sys.version.split()[0])
print("mini-SWE-agent:", minisweagent.__version__)
print("仓库根目录:", REPO)

Python: 3.13.14
mini-SWE-agent: 2.4.5
仓库根目录: /Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent


## 概念：三个对象各管什么

- **Agent**：初始化任务消息、重复 step、记录 messages、统计成本、检查限制、处理控制流、保存轨迹。**不该**知道某个模型厂商的 API 细节，也不该把 Docker 命令写死在主循环里。
- **Model**：接收消息历史、调 API、把响应解析成统一 action、把 Environment 输出格式化成模型能读的 message。
- **Environment**：接收结构化 action、决定 cwd/env、启动子进程/容器、捕获输出与返回码、处理超时、识别完成信号。

## 运行看结果：三个 Protocol 长什么样

In [2]:
# 小工具：带行号打印源码切片（相当于 nl + sed）
def show_source(rel_path: str, start: int, end: int) -> None:
    lines = (REPO / rel_path).read_text().splitlines()
    end = min(end, len(lines))
    w = len(str(end))
    for n in range(start, end + 1):
        print(f"{n:>{w}}  {lines[n - 1]}")

In [3]:
show_source("src/minisweagent/__init__.py", 43, 80)

43  class Model(Protocol):
44      """Protocol for language models."""
45  
46      config: Any
47  
48      def query(self, messages: list[dict[str, str]], **kwargs) -> dict: ...
49  
50      def format_message(self, **kwargs) -> dict: ...
51  
52      def format_observation_messages(
53          self, message: dict, outputs: list[dict], template_vars: dict | None = None
54      ) -> list[dict]: ...
55  
56      def get_template_vars(self, **kwargs) -> dict[str, Any]: ...
57  
58      def serialize(self) -> dict: ...
59  
60  
61  class Environment(Protocol):
62      """Protocol for execution environments."""
63  
64      config: Any
65  
66      def execute(self, action: dict, cwd: str = "") -> dict[str, Any]: ...
67  
68      def get_template_vars(self, **kwargs) -> dict[str, Any]: ...
69  
70      def serialize(self) -> dict: ...
71  
72  
73  class Agent(Protocol):
74      """Protocol for agents."""
75  
76      config: Any
77  
78      def run(self, task: str, **kwargs) -> dict: 

关键不是记 Protocol 语法，而是理解**“依赖能力，不依赖继承”**：
> 只要一个对象提供了 Agent 需要的方法，它就能被接入——不必先继承一个庞大的基类。

这就是 duck typing 在 Runtime 里的价值。你能随手写个 `FakeModel` / `RecordingEnvironment` / `DockerEnvironment` 接进去。

## 动手：写一个只“记录”不“执行”的 Environment

补全下面的 `RecordingEnvironment`：它不真正跑命令，只把命令记进列表。
目标是验证一个论断——**换掉 Environment，`DefaultAgent` 一行都不用改**。

In [5]:
class RecordingEnvironment:
    def __init__(self):
        self.commands = []
    def execute(self, action, cwd=""):
        # TODO: 把 action["command"] 追加进 self.commands
        self.commands.append(action["command"])
        # TODO: 返回一个形如 {"output": ..., "returncode": 0, "exception_info": ""} 的 dict
        return {"output": f"执行{action['command']}的输出", "returncode": 0, "exception_info": ""}
    def get_template_vars(self, **kwargs):
        return {}
    def serialize(self):
        return {"info": {"environment_type": "recording"}}

## 运行看结果：把它接进真实的 `DefaultAgent`（参考实现）

In [8]:
from minisweagent.agents.default import DefaultAgent
from minisweagent.models.test_models import DeterministicToolcallModel, make_toolcall_output

class RecordingEnvironmentRef:
    def __init__(self):
        self.commands = []
    def execute(self, action, cwd=""):
        self.commands.append(action["command"])
        return {"output": "recorded", "returncode": 0, "exception_info": ""}
    def get_template_vars(self, **kwargs):
        return {}
    def serialize(self):
        return {"info": {"environment_type": "recording"}}

action = {"command": "rm -rf /不会真的执行", "tool_call_id": "call_1"}
model = DeterministicToolcallModel(outputs=[make_toolcall_output("记录这个动作", [], [action])])
env = RecordingEnvironmentRef()
agent = DefaultAgent(model, env, system_template="system",
                     instance_template="{{task}}", cost_limit=5)
agent.add_messages(
    model.format_message(role="system", content="system"),
    model.format_message(role="user", content="只执行一步"),
)
agent.step()
print("记录的命令:", env.commands)
print("消息角色:", [m.get("role") for m in agent.messages])

记录的命令: ['rm -rf /不会真的执行']
消息角色: ['system', 'user', 'assistant', 'tool']


## 观察点

- `DefaultAgent` 完全不知道 `RecordingEnvironmentRef` 这个类名，`step()` 也没改。它只依赖 `execute()` 这个**能力**。
- 危险命令 `rm -rf` 没有真的执行——因为执行与否由 Environment 决定，Agent 只负责把 action 递过去。这正是“可插拔沙箱”的基础。
- `RecordingEnvironment` 永远不抛 `Submitted`，所以它带的 Agent 无法靠“提交”结束——这引出第 4 章：**能力边界 ≠ 控制流边界**。

## 连回真实源码：为什么 action 解析放在 Model 一侧

原生 tool calling 和文本代码块两种输出，解析规则不同，但 Agent 的任务完全一样：
`得到 action -> 交给 Environment -> 得到 output -> 追加 observation`。
所以 v2 把 provider-specific 的解析放进 Model，Agent 只消费统一的 `{"command": ..., "tool_call_id": ...}`。
切换解析方式时，主循环不用重写。

## 闭卷检查
- “Model 执行 Bash” 对不对？为什么？
- “Agent 解析所有厂商响应” 对不对？为什么？
- 把 `LocalEnvironment` 换成 `DockerEnvironment`，主循环该不该变？

**完成标准**：一句话说清 “Agent 管过程，Model 管模型接口与消息转换，Environment 管外部世界”，并能指出 action parsing 在 Model 一侧。